# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")

30000 rows, 44 columns


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



Lane 4 (CTR/Engagement Opportunity Scoring) maps onto a **ranking/scoring** task, not
classification or clustering. The core question is "which pages first?" — I want to order
pages by how much of an opportunity they represent, not sort them into fixed categories or
discover natural groupings. Scoring fits because the useful output is a continuous number (how
far below expected CTR a page sits) that can rank an entire inventory, letting a reviewer work
down the list until they run out of time, rather than a hard yes/no cutoff that throws away
the ordering information.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



The target is a **CTR gap score**: for each page, how far its actual CTR sits below the mean
CTR of its own position tier. This is a proxy, not a perfect measurement of "opportunity" — it
assumes the tier average is a reasonable expectation, which is a simplification I'm making on
purpose to keep the target transparent and explainable.

I'm deliberately NOT using `trend_direction` or `trend_pct` anywhere — those are leakage traps
(trend_direction is derived from trend_pct, so both are the answer in disguise, not real
features). My target is built only from `ctr` and `position_tier`, both observed, current-state
signals — not derived outcome labels calculated from something I'd later be predicting.

## 3. Success metric

*One metric you can defend. What number means 'good'?*



**Precision@K** — specifically Precision@20 and Precision@50, matching the starter pipeline's
own evaluation approach. Since a reviewer can only act on a limited number of pages, what
matters is: of the top K pages my ranking surfaces, how many are genuinely high-opportunity (a
real, meaningful CTR gap, not noise)? A generic accuracy score wouldn't reflect how this output
is actually used — nobody reviews the whole dataset, they review the top of the ranked list.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



One row = one page. Below is a real slice of the data showing exactly the columns this lane
depends on: enough visibility to matter, valid position data (`avg_position > 0`, since 0 means
"no data" not rank zero), position tier, content type, and actual CTR.

In [4]:
# The unit of analysis: one page per row, showing the columns this lane depends on
lane_slice = df[
    (df["impressions_90d"] >= 100) & (df["avg_position"] > 0)
][["content_id", "content_type", "position_tier", "avg_position",
   "impressions_90d", "ctr"]].reset_index(drop=True)

print(f"{len(lane_slice):,} pages qualify as visible with valid position data")
lane_slice.head(10)

22,006 pages qualify as visible with valid position data


,content_id,content_type,position_tier,avg_position,impressions_90d,ctr
0,content_304f48230142,keyword article,striking,10.6,3803,0.76
1,content_a1fb4e703a9e,keyword article,page_3_5,20.3,15320,0.05
2,content_9aa793d4d895,keyword article,page_3_5,36.5,12581,0.09
3,content_331d6c4de07b,keyword article,page_1,6.2,11751,0.49
4,content_d99b7a2d90ca,keyword article,page_3_5,44.0,19140,0.13
5,content_d4084a4bc775,keyword article,page_1,8.5,3970,0.03
6,content_a63219c6e95a,keyword article,page_3_5,21.2,1724,0.06
7,content_5e6c160719bc,keyword article,page_3_5,46.0,32574,0.09
8,content_c27558df2b0c,keyword article,page_1,4.9,1240,0.16
9,content_d8ee6cc6d642,keyword article,top_3,2.2,20919,1.55


In [5]:
# Sketch of the target column: CTR gap vs. the page's own position tier average
lane_slice["tier_avg_ctr"] = lane_slice.groupby("position_tier")["ctr"].transform("mean").round(4)
lane_slice["ctr_gap_score"] = (lane_slice["tier_avg_ctr"] - lane_slice["ctr"]).round(4)

# Sort to see the biggest opportunities first
lane_slice.sort_values("ctr_gap_score", ascending=False).head(10)

,content_id,content_type,position_tier,avg_position,impressions_90d,ctr,tier_avg_ctr,ctr_gap_score
6523,content_478f26883850,comparison article,page_1,7.2,118,0.0,0.3548,0.3548
21988,content_c87291853cab,comparison article,page_1,8.2,112,0.0,0.3548,0.3548
21993,content_6880eb215048,keyword article,page_1,6.8,2845,0.0,0.3548,0.3548
6474,content_268b56dc0221,keyword article,page_1,5.2,130,0.0,0.3548,0.3548
6506,content_e0ae71489787,feedly article,page_1,4.9,381,0.0,0.3548,0.3548
26,content_d87a116e2c79,keyword article,page_1,6.8,298,0.0,0.3548,0.3548
1656,content_3d2f81868b16,keyword article,page_1,5.0,497,0.0,0.3548,0.3548
18031,content_dc0d776ba8e5,keyword article,page_1,6.7,310,0.0,0.3548,0.3548
8545,content_50c7fe0d308f,comparison article,page_1,6.2,161,0.0,0.3548,0.3548
8550,content_6b5196195b13,keyword article,page_1,6.8,329,0.0,0.3548,0.3548


In [6]:
# Does the target actually vary meaningfully, or is it mostly flat around zero?
print(lane_slice["ctr_gap_score"].describe().round(4))
print(f"\n{(lane_slice['ctr_gap_score'] > 0.1).sum():,} pages have a gap over 0.10 — "
      f"a real, sizeable spread, not just noise around zero.")

count    22006.0000
mean         0.0000
std          0.3906
min        -11.4052
25%         -0.0742
50%          0.0924
75%          0.1941
max          0.3548
Name: ctr_gap_score, dtype: float64

10,710 pages have a gap over 0.10 — a real, sizeable spread, not just noise around zero.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A simple fixed rule (e.g. "flag anything under 1% CTR") doesn't work because CTR is naturally
tied to position — that rule would just flag every low-ranked page and miss real opportunities
higher up. The comparison I actually need — "is this page low *relative to its peers at the
same tier*" — already requires grouping and comparing, one step past what a single hardcoded
threshold can express cleanly.

That said, at this stage my current approach (tier-average gap) is really a **transparent
baseline rule**, not a trained model yet — which is honest and fine for now. Where a learned
model would earn its place beyond this baseline is by combining more signals at once (content
type, word count, freshness, engagement rate together) to predict a more nuanced "expected CTR"
than a single tier average can give — similar to how the starter pipeline's random forest beat
its hand-written baseline by combining more signals than a human would juggle by hand. I'll
test that comparison directly in a later assignment, once I have both a baseline and a trained
model to compare honestly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.